# 87 — Tối ưu tham số và walk-forward

`optstrategy` của backtrader chạy cả một lưới tham số chỉ bằng một dòng. Đó
vừa là tiện ích, vừa là cái bẫy: chọn bộ tham số đẹp nhất trong năm mươi bộ thì
con số đẹp đó **đã chứa sẵn may mắn** — và may mắn không đi theo bạn sang dữ
liệu mới.

Notebook này đo cái giá đó bằng số, và đo luôn một câu hỏi kỹ thuật mà ai chạy
backtrader trên Windows cũng gặp: **`maxcpus>1` có chạy được trong Jupyter
không?**

| Phần | Nội dung |
|---|---|
| 1 | Dữ liệu: rổ 10 mã chọn bằng thanh khoản **của năm đầu**, 10 năm giá |
| 2 | Chiến lược và một lần chạy |
| 3 | `optstrategy` trên lưới tham số → heatmap |
| 4 | Đa tiến trình trên Windows trong Jupyter · `optreturn` · `optdatas` |
| 5 | Walk-forward: tối ưu trong mẫu, kiểm ngoài mẫu, cửa sổ trượt |
| 6 | Vùng tham số ổn định so với đỉnh nhọn |
| 7 | Khoảng rơi — giá của overfitting |

In [1]:
import os
import sys
import time
from pathlib import Path

GOC = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "finlens_examples").is_dir())
sys.path.insert(0, str(GOC))
# Thư mục của notebook — để `import chien_luoc_87` chạy được cả trong tiến trình con
THU_MUC_08 = GOC / "notebooks" / "08-backtest-backtrader"
sys.path.insert(0, str(THU_MUC_08))

import backtrader as bt
import numpy as np
import pandas as pd
import plotly.graph_objects as go

import finlens
from finlens_examples import ap_dung_theme, duong, heatmap, hom_nay, lui_ngay
from finlens_examples.bt_vn import PhiCoPhieuVN, bo_phien_chua_dong, chi_so_hieu_qua, feed_tu_finlens, ket_qua, them_analyzer
from finlens_examples.charts import CHUOI

from chien_luoc_87 import GiaoCatSMA

ap_dung_theme()
client = finlens.client()
HOM_NAY = hom_nay(client)

D:\finlens\finlens-python\finlens-python-example\venv\Lib\site-packages\anyio\_core\_tasks.py:275: UpgradeAvailableWarning: finlens 1.5.0 đã có (đang dùng 1.4.0). Nâng cấp: pip install -U finlens
  retval = await self._coro


Mọi lựa chọn viết ra trước khi chạy. Lưới tham số được cố định **trước** khi
nhìn kết quả; nếu sửa lưới sau khi thấy heatmap thì chính việc sửa đó đã là một
vòng tối ưu nữa.

In [2]:
SO_NAM = 10
SO_MA = 10
VON = 1e9  # VND

LUOI_NHANH = [5, 10, 15, 20, 30, 40, 50]  # phiên
LUOI_CHAM = [20, 30, 40, 60, 80, 100, 120, 150, 200]  # phiên

KHOI_DONG = 250  # phiên đầu bỏ khỏi mọi phép đo: SMA chậm nhất cần 200 phiên để có giá trị
TRONG_MAU = 750  # ~3 năm để tối ưu
NGOAI_MAU = 250  # ~1 năm để kiểm, rồi trượt cả hai cửa sổ thêm 250 phiên

print(f"Lưới {len(LUOI_NHANH)} × {len(LUOI_CHAM)} = {len(LUOI_NHANH) * len(LUOI_CHAM)} tổ hợp, "
      f"{sum(n < c for n in LUOI_NHANH for c in LUOI_CHAM)} tổ hợp hợp lệ (nhanh < chậm)")

Lưới 7 × 9 = 63 tổ hợp, 54 tổ hợp hợp lệ (nhanh < chậm)


## 1 · Dữ liệu

Rổ 10 mã chọn bằng **GTGD trung vị của năm đầu tiên** trong cửa sổ 10 năm, không
phải của hôm nay — chọn bằng thanh khoản hôm nay là đem kẻ thắng của mười năm
qua vào backtest (notebook [`34`](../03-phan-tich-ky-thuat/34_backtest_chien_luoc.ipynb),
lỗi thứ nhất). Trung vị thay vì trung bình để một phiên thoả thuận lớn không
kéo một mã lên đầu.

Năm đầu tiên của cả sàn: mỗi lô 100 mã × ~250 phiên chỉ tối đa ~25.000 dòng —
dưới `max_rows`, nhưng vẫn kiểm `truncated` từng lô. Đơn vị đọc **trước** khi
ghép.

In [3]:
MA_HOSE = client.meta.symbols(exchange="HOSE", kind="stock")["symbol"].tolist()
TU_NGAY = lui_ngay(HOM_NAY, nam=SO_NAM)
lo = client.limits()["max_symbols_per_request"]

phan = []
for i in range(0, len(MA_HOSE), lo):
    d = client.eod.stock.ohlcv(MA_HOSE[i : i + lo], start=TU_NGAY, end=lui_ngay(HOM_NAY, nam=SO_NAM - 1))
    assert not d.attrs["finlens"]["truncated"], "Lô bị cắt — giảm số mã mỗi lô"
    he_so = {"kVND": 1_000.0, "VND": 1.0}[d.attrs["finlens"]["units"]["close"]]
    phan.append(d.assign(gtgd=d["close"] * he_so * d["volume"]))
nam_dau = pd.concat(phan, ignore_index=True)

gtgd_nam_dau = nam_dau.groupby("symbol", observed=True)["gtgd"].median().sort_values(ascending=False)
RO = gtgd_nam_dau.head(SO_MA).index.tolist()
print(f"Năm đầu: {nam_dau['date'].min():%d/%m/%Y} → {nam_dau['date'].max():%d/%m/%Y}, {nam_dau['symbol'].nunique()} mã")
print("Rổ:", ", ".join(f"{m} ({gtgd_nam_dau[m] / 1e9:.0f} tỷ)" for m in RO))

Năm đầu: 20/10/2016 → 20/10/2017, 319 mã
Rổ: VNM (63 tỷ), NVL (53 tỷ), PLX (35 tỷ), SBT (34 tỷ), CII (33 tỷ), HAG (32 tỷ), STB (32 tỷ), KBC (28 tỷ), MSN (27 tỷ), BID (24 tỷ)


In [4]:
gia = client.eod.stock.ohlcv(RO, start=TU_NGAY)
assert not gia.attrs["finlens"]["truncated"]
DON_VI_GIA = gia.attrs["finlens"]["units"]["close"]
gia = bo_phien_chua_dong(gia, client)
KHUNG = {m: k for m, k in gia.groupby("symbol", observed=True)}
LICH = pd.DatetimeIndex(sorted(gia["date"].unique()))
print(f"{len(gia):,} dòng · {len(LICH)} phiên · {LICH[0]:%d/%m/%Y} → {LICH[-1]:%d/%m/%Y} · giá {DON_VI_GIA}")
print("Mã niêm yết sau ngày đầu:", {m: f"{k['date'].min():%d/%m/%Y}" for m, k in KHUNG.items() if k["date"].min() > LICH[0]})

24,767 dòng · 2494 phiên · 20/10/2016 → 19/10/2026 · giá kVND
Mã niêm yết sau ngày đầu: {'NVL': '28/12/2016', 'PLX': '21/04/2017'}


## 2 · Chiến lược

`GiaoCatSMA` nằm trong file [`chien_luoc_87.py`](chien_luoc_87.py) cạnh notebook
này, **không** nằm trong một cell — lý do ở mục 4. Luật: mỗi mã giữ khi SMA nhanh
nằm trên SMA chậm, đứng ngoài khi ngược lại; mỗi mã một suất bằng 95% giá trị
danh mục chia 10, làm tròn lô 100; phí 0,15% hai chiều + thuế bán 0,1%; khớp ở
giá mở cửa phiên sau. `prenext()` gọi `next()` vì có mã niêm yết muộn
(notebook [`86`](86_danh_muc_nhieu_ma.ipynb)).

In [5]:
def tao_cerebro(**tham_so_cerebro) -> bt.Cerebro:
    cerebro = bt.Cerebro(stdstats=False, **tham_so_cerebro)
    for m in RO:
        cerebro.adddata(feed_tu_finlens(KHUNG[m], don_vi_gia=DON_VI_GIA))
    cerebro.broker.setcash(VON)
    cerebro.broker.addcommissioninfo(PhiCoPhieuVN())
    them_analyzer(cerebro)
    return cerebro


def sharpe(ls: pd.Series) -> float:
    return chi_so_hieu_qua(ls)["sharpe"]  # năm hoá, rf = 0 — so sánh giữa các tham số, không đem đi so ngoài


cerebro = tao_cerebro()
cerebro.addstrategy(GiaoCatSMA, nhanh=10, cham=50)
t0 = time.perf_counter()
mot_lan = ket_qua(cerebro.run()[0])
GIAY_MOT_LAN = time.perf_counter() - t0
ls_mot_lan = mot_lan["loi_suat"].iloc[KHOI_DONG:]
print(f"Một lần chạy: {GIAY_MOT_LAN:.1f} giây · {mot_lan['tom_tat']['so_lenh_khop']} lần khớp")
print({k: round(v, 3) for k, v in chi_so_hieu_qua(ls_mot_lan).items()})

Một lần chạy: 3.9 giây · 598 lần khớp
{'cagr': 0.091, 'mdd': -0.253, 'sharpe': 0.692, 'bien_dong': 0.14}


## 3 · `optstrategy` trên lưới tham số

`optstrategy` nhận mỗi tham số một danh sách và chạy **tích Descartes** của
chúng: 7 × 9 = 63 lần chạy. Nó không biết ràng buộc `nhanh < chậm`, nên 9 tổ hợp
vô nghĩa vẫn được chạy rồi bị bỏ ở bước đọc kết quả.

Mỗi phần tử của kết quả là một danh sách (một phần tử cho mỗi strategy đã
`optstrategy`), và mặc định (`optreturn=True`) phần tử đó **không phải
strategy** mà là một `OptReturn` chỉ mang `params` và `analyzers` — đủ để đọc
lợi suất ngày từ `TimeReturn`.

Chạy lưới một lần trên **toàn bộ** 10 năm, lấy chuỗi lợi suất ngày của từng tổ
hợp. Mọi phép đo về sau — cả kỳ, trong mẫu, ngoài mẫu — chỉ là cắt chuỗi này.

In [6]:
SO_LOI = max(2, min(8, (os.cpu_count() or 2) - 2))


def toi_uu(nhanh=LUOI_NHANH, cham=LUOI_CHAM, **tham_so_cerebro):
    cerebro = tao_cerebro(**tham_so_cerebro)
    cerebro.optstrategy(GiaoCatSMA, nhanh=nhanh, cham=cham)
    t0 = time.perf_counter()
    kq = cerebro.run()
    return kq, time.perf_counter() - t0


def bang_loi_suat(kq) -> pd.DataFrame:
    cot = {}
    for (r,) in kq:
        if r.params.nhanh < r.params.cham:
            ls = pd.Series(r.analyzers.loi_suat.get_analysis(), dtype="float64")
            cot[(r.params.nhanh, r.params.cham)] = ls
    bang = pd.DataFrame(cot)
    bang.index = pd.DatetimeIndex(bang.index)
    return bang


import warnings

with warnings.catch_warnings(record=True) as CANH_BAO_DA_LOI:
    warnings.simplefilter("always", DeprecationWarning)  # mặc định Python giấu cảnh báo phát ra ngoài __main__
    kq_luoi, GIAY_DA_LOI = toi_uu(maxcpus=SO_LOI)
LS = bang_loi_suat(kq_luoi)
print(f"{len(kq_luoi)} lần chạy trên {SO_LOI} lõi trong {GIAY_DA_LOI:.0f} giây · kiểu phần tử: {type(kq_luoi[0][0]).__name__}")
print(f"Bảng lợi suất: {LS.shape[0]} phiên × {LS.shape[1]} tổ hợp hợp lệ")
noi_dung = sorted({f"{c.category.__name__}: {c.message}" for c in CANH_BAO_DA_LOI})
print(f"{len(CANH_BAO_DA_LOI)} cảnh báo bị giấu trong lúc chạy đa lõi:")
for dong_cb in noi_dung:
    print("  " + dong_cb)

63 lần chạy trên 8 lõi trong 17 giây · kiểu phần tử: OptReturn
Bảng lợi suất: 2494 phiên × 54 tổ hợp hợp lệ
126 cảnh báo bị giấu trong lúc chạy đa lõi:


In [7]:
def thanh_luoi(gia_tri: pd.Series) -> pd.DataFrame:
    # Series theo (nhanh, cham) → bảng nhanh × chậm; ô không hợp lệ để trống
    bang = gia_tri.unstack().reindex(index=LUOI_NHANH, columns=LUOI_CHAM)
    bang.index = [f"nhanh {n}" for n in bang.index]
    bang.columns = [f"chậm {c}" for c in bang.columns]
    return bang


SHARPE_CA_KY = LS.iloc[KHOI_DONG:].apply(sharpe)
TOT_NHAT_CA_KY = SHARPE_CA_KY.idxmax()
heatmap(
    thanh_luoi(SHARPE_CA_KY),
    tieu_de="Sharpe theo tham số — cả kỳ",
    phu_de=f"{LS.index[KHOI_DONG]:%m/%Y} → {LS.index[-1]:%m/%Y} · tốt nhất: nhanh {TOT_NHAT_CA_KY[0]}, chậm {TOT_NHAT_CA_KY[1]} "
           f"(Sharpe {SHARPE_CA_KY.max():.2f}) — nhưng chọn nó là đã nhìn cả 10 năm",
    nhan_mau="Sharpe",
    dinh_dang_o="%{z:.2f}",
)

Cảnh quan khá mượt: Sharpe 0,6–0,8 ở góc SMA ngắn (nhanh 5–15, chậm 20–40) rồi
giảm đều khi SMA chậm dài ra, xuống còn 0,07–0,18 ở cột chậm 200. Ô tốt nhất cả
kỳ là **nhanh 5 / chậm 40, Sharpe 0,82** — cao hơn hẳn hai ô kề nó cùng hàng
(0,70 và 0,61).

Đó là con số một người chạy `optstrategy` một lần trên toàn bộ 10 năm sẽ trình
bày. Nó đã dùng chính 10 năm đó để chọn, nên nó **không phải** ước lượng của
những gì sẽ xảy ra tiếp theo. Mục 5–7 đo khoảng cách giữa hai thứ đó.

## 4 · Đa tiến trình trên Windows trong Jupyter

Ô trên vừa chạy lưới trên nhiều lõi — được, vì `GiaoCatSMA` **import từ một
file `.py`**. Thử điều mà hầu hết hướng dẫn backtrader viết: định nghĩa strategy
ngay trong notebook rồi `cerebro.run(maxcpus=2)`.

Thử trực tiếp ở đây thì notebook này treo, nên ta chạy nó trong một **kernel
Jupyter riêng** — cùng loại kernel với notebook này — chép nguyên lớp
`GiaoCatSMA` vào `__main__` của kernel đó như thể nó nằm trong một cell, cho
nó 45 giây, rồi đọc stderr của kernel (chỗ lỗi của tiến trình con đổ ra, không
hiện lên notebook) và dọn mọi tiến trình con.

In [8]:
import pickle
import queue
import shutil
import tempfile

import psutil
from jupyter_client import KernelManager


def chay_trong_kernel_rieng(cac_o: list[str], cho_giay: float) -> dict:
    tam = Path(tempfile.mkdtemp(dir=GOC / "output"))
    log = open(tam / "stderr.txt", "w", encoding="utf-8")
    km = KernelManager(kernel_name="python3")
    km.start_kernel(stdout=log, stderr=log, cwd=str(tam))
    kc = km.client()
    kc.start_channels()
    kc.wait_for_ready(timeout=60)
    ket_qua_o = []
    for ma_nguon in cac_o:
        msg_id, t0, in_ra, xong = kc.execute(ma_nguon), time.perf_counter(), [], False
        while time.perf_counter() - t0 < cho_giay and not xong:
            try:
                tin = kc.get_iopub_msg(timeout=1)
            except queue.Empty:
                continue
            if tin["parent_header"].get("msg_id") != msg_id:
                continue
            if tin["msg_type"] == "stream":
                in_ra.append(tin["content"]["text"])
            elif tin["msg_type"] == "error":
                in_ra.append(f"{tin['content']['ename']}: {tin['content']['evalue']}")
            elif tin["msg_type"] == "status" and tin["content"]["execution_state"] == "idle":
                xong = True
        ket_qua_o.append({"xong": xong, "giay": time.perf_counter() - t0, "in_ra": "".join(in_ra).strip()})

    pid = km.provisioner.process.pid
    con = psutil.Process(pid).children(recursive=True)
    kc.stop_channels()
    km.shutdown_kernel(now=True)
    for p in con + [p for p in psutil.process_iter(["cmdline"]) if f"parent_pid={pid}" in " ".join(p.info["cmdline"] or [])]:
        try:
            p.kill()
        except psutil.Error:
            pass
    log.close()
    stderr = (tam / "stderr.txt").read_text(encoding="utf-8", errors="replace")
    shutil.rmtree(tam, ignore_errors=True)
    return {"o": ket_qua_o, "stderr": stderr, "so_tien_trinh_con": len(con)}


tam_du_lieu = GOC / "output" / "_87_khung.pkl"
pickle.dump({"khung": KHUNG, "ro": RO, "don_vi": DON_VI_GIA}, open(tam_du_lieu, "wb"))
nguon_lop = (THU_MUC_08 / "chien_luoc_87.py").read_text(encoding="utf-8")

o_chuan_bi = f'''
import sys, pickle, time
sys.path.insert(0, {str(GOC)!r})
import backtrader as bt
from finlens_examples.bt_vn import PhiCoPhieuVN, feed_tu_finlens
exec({nguon_lop!r})          # lớp GiaoCatSMA giờ nằm trong __main__, như khi viết trong một cell
du_lieu = pickle.load(open({str(tam_du_lieu)!r}, "rb"))

def chay(maxcpus):
    cerebro = bt.Cerebro(stdstats=False, maxcpus=maxcpus)
    for m in du_lieu["ro"]:
        cerebro.adddata(feed_tu_finlens(du_lieu["khung"][m], don_vi_gia=du_lieu["don_vi"]))
    cerebro.broker.setcash(1e9)
    cerebro.optstrategy(GiaoCatSMA, nhanh=[5, 10], cham=[50, 100])
    t0 = time.perf_counter()
    kq = cerebro.run()
    print(f"maxcpus={{maxcpus}}: {{len(kq)}} kết quả trong {{time.perf_counter() - t0:.1f}} giây")

print(GiaoCatSMA.__module__, "·", __import__("multiprocessing").get_start_method())
chay(1)
'''
thu = chay_trong_kernel_rieng([o_chuan_bi, "chay(2)"], cho_giay=45)
tam_du_lieu.unlink()

for ten, o in zip(["chuẩn bị + maxcpus=1", "maxcpus=2"], thu["o"]):
    print(f"[{ten}] xong={o['xong']} sau {o['giay']:.0f} giây · in ra: {o['in_ra'] or '(không có gì)'}")
dong_loi = [d for d in thu["stderr"].splitlines() if "Error" in d]
print(f"\nstderr của kernel: {thu['stderr'].count('SpawnPoolWorker')} lần worker chết · "
      f"{thu['so_tien_trinh_con']} tiến trình con còn sống lúc dọn")
print("Dòng lỗi:", dong_loi[0] if dong_loi else "(không có)")

[chuẩn bị + maxcpus=1] xong=True sau 17 giây · in ra: __main__ · spawn
maxcpus=1: 4 kết quả trong 15.7 giây
[maxcpus=2] xong=False sau 45 giây · in ra: (không có gì)

stderr của kernel: 4 lần worker chết · 3 tiến trình con còn sống lúc dọn
Dòng lỗi: AttributeError: Can't get attribute 'GiaoCatSMA' on <module '__main__' (<class '_frozen_importlib.BuiltinImporter'>)>


**Không chạy được — và không báo lỗi.** Ô chuẩn bị xong bình thường: lớp nằm ở
`__main__`, phương thức khởi tạo tiến trình là `spawn`, và `maxcpus=1` cho đủ 4
kết quả. `maxcpus=2` thì 45 giây không in ra gì, không ngoại lệ, không kết quả.
Toàn bộ lỗi nằm ở stderr của kernel — chỗ người dùng notebook không nhìn thấy:
mỗi worker chết với cùng một dòng `AttributeError: Can't get attribute
'GiaoCatSMA' on <module '__main__'>` (4 worker chết trong 45 giây, 3 tiến trình
con còn sống phải giết lúc dọn).

Lần đầu thử điều này khi viết notebook, ô bị `nbconvert` cắt bằng
`CellTimeoutError` sau 90 giây, kernel bị giết, và hai worker còn lại thành
tiến trình mồ côi phải tìm và giết bằng tay.

**Cơ chế.** Windows không có `fork`; `multiprocessing` dùng **spawn** — mỗi
worker là một Python mới, nhận công việc dưới dạng pickle. Pickle của một lớp
chỉ là **tên** của nó: `__main__.GiaoCatSMA`. Trong worker, `__main__` là một
module rỗng (kernel Jupyter không có file script để chạy lại), nên unpickle
thất bại. Pool không coi đó là lỗi của tác vụ mà là worker chết, đẻ worker mới
thay thế, worker mới lại chết… Tiến trình chính chờ kết quả mãi mãi, và lỗi chỉ
hiện ở stderr của kernel — trong notebook không thấy gì.

**Cách xử lý:** đặt strategy (và mọi lớp/hàm nó dùng: indicator, sizer,
analyzer tự viết) vào một file `.py` rồi import. Worker khi đó chỉ cần
`import chien_luoc_87`; `sys.path` của tiến trình cha được truyền sang nên
`finlens_examples` cũng import được. Linux/macOS dùng `fork` nên không gặp lỗi
này — đó là lý do lỗi ít được nhắc trong tài liệu.

### Nhanh hơn bao nhiêu, và `optreturn` / `optdatas` làm gì

Chạy lại cả lưới trên **một** lõi để so. Rồi trên một lưới con 6 tổ hợp, bật/tắt
hai tham số tối ưu của `Cerebro`:

- `optdatas=True` — nạp dữ liệu **một lần** ở tiến trình chính rồi gửi bản đã
  nạp cho các worker, thay vì mỗi lần chạy nạp lại.
- `optreturn=True` — trả về `OptReturn` (chỉ `params` + `analyzers`) thay cho cả
  strategy với mọi line, indicator, dữ liệu — nhẹ hơn khi gửi kết quả từ worker
  về.

In [9]:
kq_mot_loi, GIAY_MOT_LOI = toi_uu(maxcpus=1)
khop = np.allclose(bang_loi_suat(kq_mot_loi).to_numpy(), LS.to_numpy(), equal_nan=True)
print(f"Cả lưới {len(kq_mot_loi)} lần chạy: 1 lõi {GIAY_MOT_LOI:.0f} giây · {SO_LOI} lõi {GIAY_DA_LOI:.0f} giây "
      f"→ nhanh gấp {GIAY_MOT_LOI / GIAY_DA_LOI:.1f} lần · kết quả giống hệt: {khop}")

LUOI_CON = dict(nhanh=[5, 10, 20], cham=[50, 100])
warnings.simplefilter("ignore", DeprecationWarning)
do = []
for maxcpus in (1, SO_LOI):
    for optdatas, optreturn in [(True, True), (False, True), (True, False)]:
        kq, giay = toi_uu(**LUOI_CON, maxcpus=maxcpus, optdatas=optdatas, optreturn=optreturn)
        do.append({
            "maxcpus": maxcpus,
            "optdatas": optdatas,
            "optreturn": optreturn,
            "giây": round(giay, 1),
            "kiểu kết quả": type(kq[0][0]).__name__,
            "pickle kết quả (KB)": round(len(pickle.dumps(kq)) / 1024),  # cảnh báo itertools đã nói ở trên
        })
TOI_UU_CO = pd.DataFrame(do)
TOI_UU_CO

Cả lưới 63 lần chạy: 1 lõi 247 giây · 8 lõi 17 giây → nhanh gấp 14.6 lần · kết quả giống hệt: True


,maxcpus,optdatas,optreturn,giây,kiểu kết quả,pickle kết quả (KB)
0,1,True,True,23.3,OptReturn,593
1,1,False,True,23.4,OptReturn,593
2,1,True,False,23.6,GiaoCatSMA,17532
3,8,True,True,6.0,OptReturn,594
4,8,False,True,7.2,OptReturn,594
5,8,True,False,6.5,GiaoCatSMA,31497


Một lượt chạy tốn thời gian vào đâu? Chạy `Cerebro` với một strategy **rỗng**
— mọi thứ còn lại là chi phí của bộ máy chứ không phải của luật giao dịch — và
bấm giờ riêng bước `preload` (nạp từng dòng của DataFrame vào các line).

In [10]:
from backtrader.feed import AbstractDataBase

preload_goc = AbstractDataBase.preload  # PandasData kế thừa preload từ lớp này
lan_nap: list[float] = []


def preload_bam_gio(self):
    t0 = time.perf_counter()
    preload_goc(self)
    lan_nap.append(time.perf_counter() - t0)


AbstractDataBase.preload = preload_bam_gio
try:
    cerebro = tao_cerebro()
    cerebro.addstrategy(bt.Strategy)  # strategy rỗng
    t0 = time.perf_counter()
    cerebro.run()
    TONG_RONG = time.perf_counter() - t0
finally:
    AbstractDataBase.preload = preload_goc

NAP = sum(lan_nap)
print(f"Strategy rỗng: {TONG_RONG:.2f} giây/lượt · trong đó preload {len(lan_nap)} feed: {NAP:.2f} giây = {NAP / TONG_RONG:.0%}")
print(f"So với một lượt có strategy thật ở mục 2: {GIAY_MOT_LAN:.1f} giây")

Strategy rỗng: 3.26 giây/lượt · trong đó preload 10 feed: 2.93 giây = 90%
So với một lượt có strategy thật ở mục 2: 3.9 giây


**8 lõi nhanh gấp 14,6 lần 1 lõi** (247 → 17 giây cho 63 lần chạy) — nhiều hơn
số lõi. Ô bấm giờ ở trên giải thích vì sao: **90% thời gian của một lượt là
`preload`**, tức nạp từng dòng DataFrame vào các line (2,93 trên 3,26 giây với
strategy rỗng; một lượt có strategy thật cũng chỉ 3,9 giây). Luật giao dịch gần
như không tốn gì; bộ máy mới tốn.

Và `optdatas` — tham số bỏ được đúng phần đó — **chỉ có tác dụng khi
`maxcpus>1`**. Trong mã nguồn `Cerebro.run`, nhánh `maxcpus == 1` gọi
`runstrategies` không kèm dữ liệu nạp sẵn nên mỗi lượt nạp lại từ đầu; chỉ nhánh
đa tiến trình mới nạp một lần ở tiến trình chính rồi gửi bản đã nạp cho worker.
Bảng con khớp với điều đó: ở 1 lõi bật hay tắt `optdatas` không đổi gì (23,3 so
với 23,4 giây), ở 8 lõi tắt nó thì chậm
hơn (7,2 so với 6,0 giây). Tăng tốc vượt số lõi = song song × bỏ được `preload`.

`optreturn=False` trả về strategy đầy đủ thay cho `OptReturn`: cùng 6 kết quả
nặng **17.532 KB** (1 lõi) và **31.497 KB** (8 lõi) thay vì 593 KB — gấp 30–53
lần, phải gửi qua pipe từ worker về rồi giữ trong RAM. Ở lưới nhỏ này thời gian
gần như không đổi; ở lưới vài nghìn tổ hợp thì đó là chuyện hết bộ nhớ. Chỉ tắt
khi thật sự cần đọc line hay indicator của từng lần chạy.

⚠️ **Cảnh báo cho tương lai.** Lần chạy 8 lõi ở mục 3 phát ra 126 cảnh báo bị
Python giấu theo mặc định — 2 cảnh báo mỗi lượt: `Pickle, copy, and deepcopy
support will be removed from itertools in Python 3.14`. Đường đa tiến trình
pickle `Cerebro`, và `Cerebro` giữ đối tượng `itertools`. Theo chính lời cảnh
báo, trên Python 3.14 việc pickle đó sẽ không còn được hỗ trợ — mà backtrader
đã ngừng phát triển. Venv của dự án đang là 3.12, nên hôm nay vẫn chạy.

## 5 · Walk-forward

Cửa sổ trượt: tối ưu trên 750 phiên (**trong mẫu**), lấy bộ tham số có Sharpe
cao nhất, đo nó trên 250 phiên tiếp theo (**ngoài mẫu**) mà nó chưa từng thấy;
trượt cả hai cửa sổ thêm 250 phiên rồi lặp lại.

Mọi cửa sổ đều cắt từ bảng `LS` của mục 3 — một lần `optstrategy` cho cả 10 năm
thay vì một lần cho mỗi cửa sổ. Được phép vì SMA chỉ nhìn về quá khứ: lợi suất
của phiên `t` không phụ thuộc gì vào dữ liệu sau `t`. Khác biệt duy nhất so với
chạy lại từ đầu mỗi cửa sổ là vị thế mang sang đầu cửa sổ — và chính cái đó mới
giống đời thực, nơi chiến lược không khởi động lại vào ngày 1/1. Kiểm bằng số ở
ô sau bảng.

In [11]:
CUA_SO = []
i = KHOI_DONG
while i + TRONG_MAU < len(LS):
    ngoai = LS.iloc[i + TRONG_MAU : i + TRONG_MAU + NGOAI_MAU]
    if len(ngoai) < NGOAI_MAU // 2:
        break
    CUA_SO.append((LS.iloc[i : i + TRONG_MAU], ngoai))
    i += NGOAI_MAU

dong = []
for trong, ngoai in CUA_SO:
    s_trong, s_ngoai = trong.apply(sharpe), ngoai.apply(sharpe)
    tot = s_trong.idxmax()
    dong.append({
        "trong mẫu": f"{trong.index[0]:%m/%Y}–{trong.index[-1]:%m/%Y}",
        "ngoài mẫu": f"{ngoai.index[0]:%m/%Y}–{ngoai.index[-1]:%m/%Y}",
        "tốt nhất trong mẫu": f"{tot[0]}/{tot[1]}",
        "Sharpe trong mẫu": s_trong[tot],
        "Sharpe ngoài mẫu": s_ngoai[tot],
        "hạng ngoài mẫu": f"{int(s_ngoai.rank(ascending=False)[tot])}/{len(s_ngoai)}",
        "trung vị lưới — trong": s_trong.median(),
        "trung vị lưới — ngoài": s_ngoai.median(),
        "tương quan hạng trong↔ngoài": s_trong.rank().corr(s_ngoai.rank()),
    })
WF = pd.DataFrame(dong)
WF.round(2)

,trong mẫu,ngoài mẫu,tốt nhất trong mẫu,Sharpe trong mẫu,Sharpe ngoài mẫu,hạng ngoài mẫu,trung vị lưới — trong,trung vị lưới — ngoài,tương quan hạng trong↔ngoài
0,10/2017–10/2020,10/2020–10/2021,10/30,0.86,1.04,53/54,-0.06,1.48,-0.80
1,10/2018–10/2021,10/2021–10/2022,10/20,0.92,0.63,6/54,0.58,0.05,0.31
2,10/2019–10/2022,10/2022–10/2023,5/40,1.21,0.90,1/54,0.56,0.37,0.75
3,10/2020–10/2023,10/2023–10/2024,5/40,1.19,0.12,8/54,0.69,-0.14,-0.15
4,10/2021–10/2024,10/2024–10/2025,5/40,0.78,1.88,7/54,0.07,1.49,0.18
5,10/2022–10/2025,10/2025–10/2026,5/40,1.02,-1.15,41/54,0.60,-1.04,0.43


In [12]:
# Kiểm cách cắt: chạy lại TỪ ĐẦU tham số tốt nhất của cửa sổ cuối, chỉ trên dữ liệu
# từ (đầu ngoài mẫu − KHOI_DONG phiên) tới hết ngoài mẫu, rồi so Sharpe ngoài mẫu.
trong, ngoai = CUA_SO[-1]
tot = trong.apply(sharpe).idxmax()
tu = LS.index[LS.index.get_loc(ngoai.index[0]) - KHOI_DONG]

cerebro = bt.Cerebro(stdstats=False)
for m in RO:
    cerebro.adddata(feed_tu_finlens(KHUNG[m], don_vi_gia=DON_VI_GIA, fromdate=tu, todate=ngoai.index[-1]))
cerebro.broker.setcash(VON)
cerebro.broker.addcommissioninfo(PhiCoPhieuVN())
them_analyzer(cerebro)
cerebro.addstrategy(GiaoCatSMA, nhanh=tot[0], cham=tot[1])
chay_lai = ket_qua(cerebro.run()[0])["loi_suat"].loc[ngoai.index[0] :]

print(f"Tham số {tot[0]}/{tot[1]}, ngoài mẫu {ngoai.index[0]:%d/%m/%Y}–{ngoai.index[-1]:%d/%m/%Y}")
print(f"  cắt từ bảng LS: Sharpe {sharpe(ngoai[tot]):.3f}")
print(f"  chạy lại từ đầu: Sharpe {sharpe(chay_lai):.3f}  (tương quan lợi suất ngày {ngoai[tot].corr(chay_lai):.3f})")

Tham số 5/40, ngoài mẫu 24/10/2025–19/10/2026
  cắt từ bảng LS: Sharpe -1.154
  chạy lại từ đầu: Sharpe -1.163  (tương quan lợi suất ngày 1.000)


Cách cắt từ một lần chạy cả kỳ đúng như kỳ vọng: chạy lại từ đầu ra Sharpe
ngoài mẫu −1,163 so với −1,154, tương quan lợi suất ngày 1,000 — chỉ khác ở vài
phiên đầu khi vị thế còn đang được dựng.

Đọc bảng walk-forward:

- Ở cửa sổ đầu, tham số tốt nhất trong mẫu (10/30) **đứng thứ 53/54** ngoài mẫu.
  Ở cửa sổ thứ ba, 5/40 đứng thứ 1/54. Không có quy luật nào ổn định giữa hai
  thái cực đó.
- **Tương quan hạng** giữa Sharpe trong mẫu và ngoài mẫu trên cả lưới đi từ
  −0,80 tới +0,75. Một con số âm nghĩa là năm sau đảo ngược thứ hạng của ba năm
  trước — tham số "tốt" thành tham số "xấu".
- Trung vị lưới thay đổi dữ dội giữa các cửa sổ (−1,04 tới +1,49 ngoài mẫu): phần
  lớn biến động của Sharpe đến từ **thị trường của năm đó**, không phải từ tham
  số.

## 6 · Vùng ổn định so với đỉnh nhọn

Một bộ tham số tốt **thật** hiếm khi đứng một mình: nếu `nhanh 10 / chậm 40`
tốt vì một lý do có thật về thị trường thì `nhanh 15 / chậm 40` cũng phải gần tốt
như vậy. Một đỉnh nhọn giữa những ô xấu là dấu hiệu của may mắn.

Đo bằng **trung bình láng giềng**: mỗi ô thay bằng trung bình Sharpe của ô đó và
các ô kề nó (khối 3×3 trên lưới, bỏ ô không hợp lệ). Chọn tham số theo giá trị
đã làm mịn thay vì giá trị thô.

In [13]:
def lam_min(gia_tri: pd.Series) -> pd.Series:
    bang = gia_tri.unstack().reindex(index=LUOI_NHANH, columns=LUOI_CHAM).to_numpy()
    ra = {}
    for i, n in enumerate(LUOI_NHANH):
        for j, c in enumerate(LUOI_CHAM):
            if not np.isnan(bang[i, j]):
                ra[(n, c)] = np.nanmean(bang[max(0, i - 1) : i + 2, max(0, j - 1) : j + 2])
    return pd.Series(ra)


heatmap(
    thanh_luoi(lam_min(SHARPE_CA_KY)),
    tieu_de="Sharpe trung bình láng giềng — cả kỳ",
    phu_de="Mỗi ô = trung bình khối 3×3 quanh nó · vùng sáng rộng là vùng ổn định",
    nhan_mau="Sharpe",
    dinh_dang_o="%{z:.2f}",
)

In [14]:
dong = []
for k, (trong, ngoai) in enumerate(CUA_SO):
    s_trong, s_ngoai = trong.apply(sharpe), ngoai.apply(sharpe)
    min_trong = lam_min(s_trong)
    dinh, vung = s_trong.idxmax(), min_trong.idxmax()
    dong.append({
        "ngoài mẫu": f"{ngoai.index[0]:%m/%Y}–{ngoai.index[-1]:%m/%Y}",
        "đỉnh": f"{dinh[0]}/{dinh[1]}",
        "đỉnh — trong": s_trong[dinh],
        "đỉnh — ngoài": s_ngoai[dinh],
        "độ nhọn của đỉnh": s_trong[dinh] - min_trong[dinh],
        "vùng": f"{vung[0]}/{vung[1]}",
        "vùng — trong": s_trong[vung],
        "vùng — ngoài": s_ngoai[vung],
    })
ON_DINH = pd.DataFrame(dong)
ON_DINH.round(2)

,ngoài mẫu,đỉnh,đỉnh — trong,đỉnh — ngoài,độ nhọn của đỉnh,vùng,vùng — trong,vùng — ngoài
0,10/2020–10/2021,10/30,0.86,1.04,0.16,15/20,0.65,1.29
1,10/2021–10/2022,10/20,0.92,0.63,0.13,15/20,0.73,0.30
2,10/2022–10/2023,5/40,1.21,0.90,0.30,5/30,0.85,0.89
3,10/2023–10/2024,5/40,1.19,0.12,0.26,5/30,0.91,0.03
4,10/2024–10/2025,5/40,0.78,1.88,0.26,5/30,0.64,1.92
5,10/2025–10/2026,5/40,1.02,-1.15,0.15,5/30,1.00,-1.15


Đỉnh trong mẫu luôn nhô lên khỏi vùng xung quanh 0,13–0,30 điểm Sharpe — phần
nhô đó là thứ mà giá trị làm mịn bỏ đi. Cách chọn theo vùng ổn định đổi tham số
ít hơn (15/20 hai cửa sổ đầu, 5/30 bốn cửa sổ sau) và luôn nằm sát cạnh đỉnh
trên lưới.

Nhưng ngoài mẫu, **vùng ổn định không thắng đỉnh**: mỗi bên thắng 3/6 cửa sổ.
Ở đây cảnh quan Sharpe đủ mượt (heatmap thứ hai) để đỉnh và vùng quanh nó đi
lên đi xuống cùng nhau; lợi ích của làm mịn sẽ lớn hơn trên một cảnh quan lởm
chởm — nhiều tham số hơn, ít dữ liệu hơn. Điều nó làm được ở đây nằm ở mục sau:
nó cho một ước lượng trong mẫu **trung thực hơn**.

## 7 · Khoảng rơi — giá của overfitting

Ba cách chọn tham số, cùng một loạt cửa sổ:

- **Đỉnh** — Sharpe trong mẫu cao nhất.
- **Vùng ổn định** — trung bình láng giềng trong mẫu cao nhất.
- **Không chọn** — trung vị của cả lưới: hiệu quả của một bộ tham số "điển hình".

Khoảng rơi của cách "không chọn" là phần do **thị trường đổi chế độ** giữa hai
cửa sổ — không ai tránh được. Phần rơi *thêm* của cách "đỉnh" so với nó là
**thiên lệch do chọn**: Sharpe trong mẫu của đỉnh cao vì được chọn là cao nhất
trong 54 con số nhiễu.

In [15]:
KHOANG_ROI = pd.DataFrame(
    {
        "Sharpe trong mẫu (tb)": [WF["Sharpe trong mẫu"].mean(), ON_DINH["vùng — trong"].mean(), WF["trung vị lưới — trong"].mean()],
        "Sharpe ngoài mẫu (tb)": [WF["Sharpe ngoài mẫu"].mean(), ON_DINH["vùng — ngoài"].mean(), WF["trung vị lưới — ngoài"].mean()],
    },
    index=["Đỉnh", "Vùng ổn định", "Không chọn (trung vị lưới)"],
)
KHOANG_ROI["khoảng rơi"] = KHOANG_ROI["Sharpe trong mẫu (tb)"] - KHOANG_ROI["Sharpe ngoài mẫu (tb)"]
THIEN_LECH_CHON = KHOANG_ROI.loc["Đỉnh", "khoảng rơi"] - KHOANG_ROI.loc["Không chọn (trung vị lưới)", "khoảng rơi"]
print(f"{len(CUA_SO)} cửa sổ · thiên lệch do chọn đỉnh = {THIEN_LECH_CHON:.2f} điểm Sharpe · "
      f"tương quan hạng trong↔ngoài trung bình {WF['tương quan hạng trong↔ngoài'].mean():+.2f}")
print(f"Ngoài mẫu, đỉnh thắng vùng ổn định ở {(ON_DINH['đỉnh — ngoài'] > ON_DINH['vùng — ngoài']).sum()}/{len(ON_DINH)} cửa sổ")
KHOANG_ROI.round(2)

6 cửa sổ · thiên lệch do chọn đỉnh = 0.39 điểm Sharpe · tương quan hạng trong↔ngoài trung bình +0.12
Ngoài mẫu, đỉnh thắng vùng ổn định ở 3/6 cửa sổ


,Sharpe trong mẫu (tb),Sharpe ngoài mẫu (tb),khoảng rơi
Đỉnh,1.00,0.57,0.43
Vùng ổn định,0.80,0.55,0.25
Không chọn (trung vị lưới),0.41,0.37,0.04


Cùng câu chuyện, viết bằng đường vốn: nối các đoạn ngoài mẫu của từng cách
chọn, đặt cạnh đường vốn của **tham số tốt nhất cả kỳ** — bộ tham số mà một
người chỉ chạy `optstrategy` một lần trên toàn bộ dữ liệu sẽ trình bày.

In [16]:
def noi_ngoai_mau(chon_tham_so) -> pd.Series:
    doan = []
    for trong, ngoai in CUA_SO:
        doan.append(ngoai[chon_tham_so(trong.apply(sharpe))])
    return pd.concat(doan)


noi = {
    "Đỉnh (walk-forward)": noi_ngoai_mau(lambda s: s.idxmax()),
    "Vùng ổn định (walk-forward)": noi_ngoai_mau(lambda s: lam_min(s).idxmax()),
    f"Tốt nhất cả kỳ {TOT_NHAT_CA_KY[0]}/{TOT_NHAT_CA_KY[1]} (nhìn trước)": LS[TOT_NHAT_CA_KY].loc[CUA_SO[0][1].index[0] :],
}
ve = pd.concat(
    [pd.DataFrame({"date": s.index, "von": ((1 + s).cumprod() * 100).to_numpy(), "duong": ten}) for ten, s in noi.items()]
)
print(pd.DataFrame({ten: chi_so_hieu_qua(s) for ten, s in noi.items()}).T[["cagr", "sharpe", "mdd"]].round(3).to_string())
duong(
    ve,
    x="date",
    y="von",
    theo="duong",
    tieu_de="Đường vốn ngoài mẫu",
    phu_de="Mỗi đoạn một năm, dùng tham số chọn bằng 3 năm trước đó · đường thứ ba biết trước cả 10 năm",
    nhan_y="giá trị danh mục (gốc = 100)",
)

                                  cagr  sharpe    mdd
Đỉnh (walk-forward)               0.09   0.687 -0.251
Vùng ổn định (walk-forward)       0.09   0.672 -0.270
Tốt nhất cả kỳ 5/40 (nhìn trước)  0.12   0.876 -0.232


**Khoảng rơi của đỉnh là 0,43 điểm Sharpe** (1,00 trong mẫu → 0,57 ngoài mẫu),
trong khi một bộ tham số "điển hình" chỉ rơi 0,04. Hiệu 0,39 điểm Sharpe là
**thiên lệch do chọn** — phần Sharpe chỉ tồn tại vì ta đã chọn số lớn nhất
trong 54 con số. Nó bằng khoảng hai phần ba Sharpe ngoài mẫu còn lại.

Vùng ổn định rơi 0,25 (0,80 → 0,55): không kiếm thêm được gì ngoài mẫu so với
đỉnh (0,55 so với 0,57), nhưng con số trong mẫu của nó **hứa ít hơn và giữ lời
hơn**. Nếu phải báo một con số trước khi có dữ liệu mới, đó là con số nên báo.

Trên đường vốn: walk-forward cho CAGR ~9%/năm, Sharpe 0,67–0,69, cả hai cách
chọn như nhau; bộ tham số tốt nhất cả kỳ (5/40), áp lên cùng khoảng ngoài mẫu,
cho 12%/năm, Sharpe 0,88. Khoảng cách đó là phần lợi nhuận **không thể có**
nếu không biết trước 10 năm.

| Đo được | Con số |
|---|---|
| Thiên lệch do chọn đỉnh trong 54 tổ hợp | 0,39 điểm Sharpe |
| Tương quan hạng trong ↔ ngoài mẫu, trung bình 6 cửa sổ | +0,12 |
| `maxcpus=2`, strategy định nghĩa trong notebook | treo, lỗi chỉ ở stderr của kernel |
| `maxcpus=8`, strategy trong `chien_luoc_87.py` | nhanh gấp 14,6 lần 1 lõi, kết quả giống hệt |
| Phần thời gian của một lượt nằm ở `preload` | 90% |
| `optdatas` ở `maxcpus=1` | không có tác dụng |
| `optreturn=False` | kết quả nặng gấp 30–53 lần |

**Quy tắc rút ra:**

1. Strategy, indicator, sizer tự viết → file `.py` cạnh notebook. Không ngoại lệ
   nếu sẽ chạy đa lõi trên Windows.
2. Tối ưu thì luôn kèm walk-forward, và báo con số **ngoài mẫu**.
3. Báo khoảng rơi cạnh con số tối ưu. Một Sharpe trong mẫu không kèm khoảng rơi
   là một con số chưa xong.

---

Tiếp theo: [`88_phai_sinh_va_intraday.ipynb`](88_phai_sinh_va_intraday.ipynb) —
VN30F1M, hệ số nhân, ký quỹ, và dữ liệu intraday.